# M4 Assignment2 - Green Patent Detection (PatentSBERTa): Active Learning + LLM→Human HITL
>Part C: Implement LLM → Human HITL (Gold Labels)
- LLM step: given only the claim text, output: llm_green_suggested (0/1), llm_confidence (low/medium/high), and llm_rationale (1–3 sentences; cite phrases from the claim).

In [1]:
!pip install pandas langchain langchain-community


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


STEP01: Load Libraries

In [2]:
import pandas as pd
import re
import time

from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate


c:\Users\Jonas Pedesk\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


STEP02: Load data from part B

In [3]:
df = pd.read_csv("hitl_green_100.csv")

df["llm_green_suggested"] = None
df["llm_confidence"] = None
df["llm_rationale"] = None
df["is_green_human"] = None
df["human_notes"] = None

STEP03: Set up LLM (Using Ollama Gemma3:4b) and struture the prompt

In [4]:
llm = ChatOllama(
    model="gemma3:4b",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
You are a patent analyst.

Based ONLY on the claim text below, determine whether it describes green technology related to sustainability, renewable energy, emission reduction, climate mitigation, or environmental protection.

Claim:
\"\"\"{text}\"\"\"

Respond strictly in this format:

llm_green_suggested: 0 or 1
llm_confidence: low / medium / high
llm_rationale: 1-3 sentences explaining why, citing phrases from the claim.
""")

chain = prompt | llm

C:\Users\Jonas Pedesk\AppData\Local\Temp\ipykernel_24944\1183899073.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(


STEP04: LLM evaluation in loop

In [6]:
for i, row in df.iterrows():

    response = chain.invoke({"text": row["text"]})
    output = response.content

    # Parse structured output
    label_match = re.search(r"llm_green_suggested:\s*(\d)", output)
    confidence_match = re.search(r"llm_confidence:\s*(low|medium|high)", output, re.IGNORECASE)
    rationale_match = re.search(r"llm_rationale:\s*(.*)", output, re.DOTALL)

    df.loc[i, "llm_green_suggested"] = int(label_match.group(1)) if label_match else None
    df.loc[i, "llm_confidence"] = confidence_match.group(1).lower() if confidence_match else None
    df.loc[i, "llm_rationale"] = rationale_match.group(1).strip() if rationale_match else output

    print(f"Processed {i+1}/{len(df)}")

Processed 1/100
Processed 2/100
Processed 3/100
Processed 4/100
Processed 5/100
Processed 6/100
Processed 7/100
Processed 8/100
Processed 9/100
Processed 10/100
Processed 11/100
Processed 12/100
Processed 13/100
Processed 14/100
Processed 15/100
Processed 16/100
Processed 17/100
Processed 18/100
Processed 19/100
Processed 20/100
Processed 21/100
Processed 22/100
Processed 23/100
Processed 24/100
Processed 25/100
Processed 26/100
Processed 27/100
Processed 28/100
Processed 29/100
Processed 30/100
Processed 31/100
Processed 32/100
Processed 33/100
Processed 34/100
Processed 35/100
Processed 36/100
Processed 37/100
Processed 38/100
Processed 39/100
Processed 40/100
Processed 41/100
Processed 42/100
Processed 43/100
Processed 44/100
Processed 45/100
Processed 46/100
Processed 47/100
Processed 48/100
Processed 49/100
Processed 50/100
Processed 51/100
Processed 52/100
Processed 53/100
Processed 54/100
Processed 55/100
Processed 56/100
Processed 57/100
Processed 58/100
Processed 59/100
Proces

Save the LLM evaluation to CSV file and continue on colab

In [ ]:
df.to_csv("hitl_green_100_llm_evaluation.csv", index=False)

print("LLM evaluation complete")

LLM evaluation complete


: 